In [1]:
import findspark, pyspark

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.functions import when, col
from pyspark.sql.types import NumericType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, CountVectorizer, VectorSizeHint
from pyspark.ml.classification import LinearSVC, LinearSVCSummary
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import ParamGridBuilder


In [2]:
# Create Spark1 session: we use the following configurations:

# - master("local[*]"): Run Spark locally with as many worker threads as logical cores on your machine.
# - config("spark.driver.memory", "1g"): Allocate 1 GB of memory to the Spark driver.
# - config("spark.executor.memory", "2g"): Allocate 2 GB of memory to each Spark executor.
# - appName("IntroToSpark"): Name the Spark application "IntroToSpark".

spark = (SparkSession.builder
    .master("local[*]")
    .appName("IntroToSpark1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    # Optional: let Spark pick a free UI port instead of hardcoding 4040
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "0")
    .config("spark.driver.memory", "2g")
    .getOrCreate())

sc1 = spark.sparkContext  # To access SparkContext if needed
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/17 14:46:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet('../Cleaned_data/biweekly_basetable_w_sentiment_5.parquet')
df.show()

# -----------
# DROP COLS
# -----------
cols_to_drop = []
basetable = df.drop(*cols_to_drop)


25/12/17 14:46:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+---------------+--------------+------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

# PREPROCESSING

In [4]:
# -----------
# SPLIT DATA
# -----------
max_index = basetable.agg(F.max("biweekly_index")).collect()[0][0]
cutoff_index = int(max_index * 0.67)

train_df = basetable.filter(F.col("biweekly_index") <= cutoff_index)
test_df  = basetable.filter(F.col("biweekly_index") > cutoff_index)

# -----------
# HANDLE MISSING VALUES 
# -----------
numeric_cols = [c for c, t in train_df.dtypes if isinstance(train_df.schema[c].dataType, NumericType)]

# Remove rows for which no previous return 
train_df = train_df.filter(F.col("prev_period_return").isNotNull())
# Fill all other columns with 0
train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# -----------
# SELECT COLUMNS 
# -----------
label_col = "financing_next_period"
cat_cols = ["industry", "maturity", "locations"]
id_cols =  ["ticker", "biweekly_index","biweekly_period", "longname", "benchmark", "period_start", "period_end"]
exclude_cols = set(id_cols + [label_col])
vec_cols = ["avg_finbert_vec_comp","finbert_embedding_ft"]

feature_columns = [c for c in train_df.columns if c not in exclude_cols]
numeric_feature_cols = [col for col in feature_columns if col not in cat_cols]

# filtering based on correlations
real_numeric_feature_cols = [col for col in numeric_feature_cols if col not in vec_cols]
non_relevant_num_cols = [
    "period_open", "period_close", "period_high", "period_low", # period cols (avg_close, period_return)
    "avg_intraday_range", "max_intraday_range", # return_volatility
    "total_volume", "max_daily_volume", "avg_volume_up_days", "avg_volume_down_days", # avg_daily_volume, volume_volatility
    "prev_avg_daily_volume", # prev_volume_volatility
    "total_words_ft", # article_count_ft
    "total_words_tijd", # article_count_tijd
    "median_roberta_ca", "min_roberta_ca", "max_roberta_ca", # simplify sentiment signals
    "min_ollama_comp", "max_ollama_comp", "avg_num_words_title_comp", "avg_num_words_content_comp" # simplify sentiment signals 
]
relevant_num_cols = [c for c in real_numeric_feature_cols if c not in non_relevant_num_cols]

sent_cols = [c for c in basetable.columns if c.endswith(("_ca", "_comp", "_ft", "_tijd")) and c not in non_relevant_num_cols]

# APPLY: to train and test data
train_df[sent_cols]
test_df[sent_cols]

# -----------
# SCALING 
# -----------
scale_cols = [
    "period_open","period_close","period_high","period_low","avg_close",
    "price_volatility","avg_intraday_range","max_intraday_range",
    "total_volume","avg_daily_volume","max_daily_volume","volume_volatility",
    "avg_volume_up_days","avg_volume_down_days",
    "avg_gap_size","max_gap",
    "avg_prChg",
    "prev_price_volatility","prev_avg_daily_volume","prev_volume_volatility"
]
scale_cols = [c for c in scale_cols if c not in non_relevant_num_cols]
no_scale_cols = [c for c in relevant_num_cols if c not in scale_cols and c not in sent_cols]

# Fit stats op TRAIN
agg_exprs = []
for c in scale_cols:
    agg_exprs += [
        F.avg(F.col(c)).alias(f"{c}__mean"),
        F.stddev_samp(F.col(c)).alias(f"{c}__std"),
    ]

stats_df = train_df.groupBy("ticker").agg(*agg_exprs)

# COMPUTE z-score
def apply_per_ticker_zscore(df, stats_df, scale_cols):
    df = df.join(stats_df, on="ticker", how="left")
    zcols = []

    for c in scale_cols:
        mu = F.col(f"{c}__mean")
        sd = F.col(f"{c}__std")
        z  = f"{c}__z"

        df = df.withColumn(
            z,
            F.when(sd.isNull() | (sd == 0), F.lit(0.0))
             .otherwise((F.col(c) - mu) / sd)
        )
        zcols.append(z)

    # drop stats cols
    drop_cols = [f"{c}__mean" for c in scale_cols] + [f"{c}__std" for c in scale_cols]
    df = df.drop(*drop_cols)

    return df, zcols


train_s, zcols = apply_per_ticker_zscore(train_df, stats_df, scale_cols)
test_s,  _     = apply_per_ticker_zscore(test_df,  stats_df, scale_cols)

# SENTIMENT: scaling
sent_scaler = StandardScaler(
    inputCol="sent_vec",
    outputCol="sent_scaled",
    withStd=True,
    withMean=False   # keep sparse-friendly; sentiment is dense but this is safe
)

# -----------
# ENCODING
# -----------
# Indexers
industry_indexer = StringIndexer(inputCol="industry", outputCol="industry_idx", handleInvalid="keep")
maturity_indexer = StringIndexer(inputCol="maturity", outputCol="maturity_idx", handleInvalid="keep")

# One-hot encoder
ENC = OneHotEncoder(
    inputCols=["industry_idx", "maturity_idx"],
    outputCols=["industry_ohe", "maturity_ohe"],
    handleInvalid="keep"
)

CV = CountVectorizer(
    inputCol="locations",
    outputCol="locations_vec",
    binary=True,     # True = presence/absence; set False if counts matter
    minDF=2          # ignore super-rare locations (tune)
)

# -----------
# VECTOR ASSEMBLER 
# -----------
VA = VectorAssembler(inputCols=feature_columns, outputCol="features")

sent_VA = VectorAssembler(
    inputCols=sent_cols,
    outputCol="sent_vec",
    handleInvalid="keep"
)

VA_final = VectorAssembler(
    inputCols=["sent_scaled"] + [f"{c}_ohe" for c in ["industry", "maturity"]] + ["locations_vec"] + zcols + no_scale_cols,
    #  ^^^ adjust these to whatever you currently assemble
    outputCol="features"
)

# -----------
# GIVE WEIGHTS TO THE LABELS
# -----------
counts = (
    train_df
    .groupBy(label_col)
    .count()
    .withColumnRenamed("count", "n")
)

total = counts.agg(F.sum("n")).collect()[0][0]

weights = (
    counts
    .withColumn("weight", total / (2 * F.col("n")))
)

def apply_weights(df):
    return df.join(weights, on="financing_next_period", how="left")

train_w = apply_weights(train_s)
test_w  = apply_weights(test_s)

# -----------
# FIX VECTORS
# -----------
EMB_DIM = 768
vector_cols = [
    "avg_finbert_vec_comp",
    "finbert_embedding_ft"
]
hints = [
    VectorSizeHint(
        inputCol=c,
        size=768,
        handleInvalid="error"   # or "skip"
    )
    for c in vector_cols
]

In [5]:
train_w.show()

+---------------------+------+--------+---------------+--------------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+---------

# MODELING

In [6]:
threshold_list = [i * 0.25 for i in range(-12, 13)]

# -----------
# DEFINE FUNCTION: a function that calculates all the evaluation metrics
# -----------
def calculate_classification_metrics(predictions, label_col, prediction_col):
    # calculate true positives, true negatives, false positives, false negatives
    tp = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).count()
    tn = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).count()
    fp = predictions.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).count()
    fn = predictions.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).count()

    # calculate accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # calculate precision
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0  
    
    # calculate recall
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0  
    
    # calculate F1 measure
    f1_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0  

    # print the metrics
    #print(f"Accuracy: {accuracy}")
    #print(f"Precision: {precision}")
    #print(f"Recall: {recall}")
    #print(f"F1 measure: {f1_measure}")
    
    return accuracy, precision, recall, f1_measure

# -----------
# DEFINE FUNCTION: a function that makes custom predictions based on a given threshold
# -----------    
def custom_predictions(threshold, df):
    df = df.withColumn(
        "rawPredictionArray",
        vector_to_array(F.col("rawPrediction"))
    ).withColumn(
        "custom_prediction",
        F.when(F.col("rawPredictionArray")[1] >= threshold, 1).otherwise(0)
    )
    return df
    

def optimize_threshold(thresholds, raw_predictions ,cost_accuracy = 1, cost_precision = 1, cost_recall = 1, cost_f1 = 1):
    # create evaluation frame
    total_cost = []
    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []

    cut_off_values = []
    for threshold in thresholds:
        evaluation_frame = custom_predictions(threshold, raw_predictions)
        accuracy, precision, recall, f1_measure = calculate_classification_metrics(evaluation_frame, 'financing_next_period', 'custom_prediction') 
        total_cost_i = accuracy*cost_accuracy + precision*cost_precision + recall*cost_recall + f1_measure*cost_f1
        total_cost.append(total_cost_i)

        accuracy_list.append(accuracy)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1_measure)
        cut_off_values.append(threshold)

    max_profit = max(total_cost)
    max_index = total_cost.index(max_profit)
    best_cut_off = cut_off_values[max_index]

    best_accuracy = accuracy_list[max_index]
    best_precision = precision_list[max_index]
    best_recall = recall_list[max_index]
    best_f1 = f1_list[max_index]

    print(f'The best threshold is {best_cut_off}, with a score of {max_profit}')
    print(f"Accuracy: {best_accuracy}")
    print(f"Precision: {best_precision}")
    print(f"Recall: {best_recall}")
    print(f"F1 measure: {best_f1}")

    return best_cut_off

In [7]:
# -----------
# SVM
# -----------
# Step 1: Define SVM model
svm = LinearSVC(featuresCol="features", labelCol="financing_next_period", maxIter=100, regParam=0.1)

# Step 2: Create a pipeline to include preprocessing and SVM model
svm_pipeline = Pipeline(stages=[
    industry_indexer,  # Index categorical column 'industry'
    maturity_indexer,  # Index categorical column 'maturity'
    ENC,               # One-hot encode index columns
    CV,                # Count vectorize 'locations' feature           
    *hints,
    sent_VA, 
    sent_scaler, 
    VA_final,
    svm                # Train Linear SVM
])

# Step 3: Train the model
svm_model = svm_pipeline.fit(train_w)  # `train_s` contains standardized training data

# Step 4: Make predictions
svm_predictions = svm_model.transform(test_w)  # `test_s` contains standardized test data

# Step 5: Evaluate the model
svm_predictions.select('ticker', 'biweekly_period', 'financing_next_period', 'prediction', 'rawPrediction').show(truncate=False)

25/12/17 14:46:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+------+---------------+---------------------+----------+------------------------------------------+
|ticker|biweekly_period|financing_next_period|prediction|rawPrediction                             |
+------+---------------+---------------------+----------+------------------------------------------+
|ARG.TO|P31            |0                    |0.0       |[1.1408528345467532,-1.1408528345467532]  |
|ARG.TO|P32            |0                    |0.0       |[2.2391040624489533,-2.2391040624489533]  |
|ARG.TO|P33            |0                    |0.0       |[1.2495625597510176,-1.2495625597510176]  |
|ARG.TO|P34            |0                    |0.0       |[1.8472194579888779,-1.8472194579888779]  |
|ARG.TO|P35            |0                    |0.0       |[1.6902768845683709,-1.6902768845683709]  |
|ARG.TO|P36            |0                    |0.0       |[2.8387262545936442,-2.8387262545936442]  |
|ARG.TO|P37            |0                    |0.0       |[0.9440914796362541,-0.94409147963

In [8]:
eval_df = svm_predictions.select('ticker', 'biweekly_period', 'financing_next_period', 'prediction', 'rawPrediction')

# Optimize threshold
optimize_threshold(threshold_list, eval_df)
# get AUC
evaluator = BinaryClassificationEvaluator(labelCol="financing_next_period", 
                                          rawPredictionCol="rawPrediction", 
                                          metricName="areaUnderROC")

lr_AUC = evaluator.evaluate(svm_predictions)

evaluator_auc_pr = BinaryClassificationEvaluator(
    labelCol="financing_next_period", 
    metricName="areaUnderPR"
)
svm_auc_pr = evaluator_auc_pr.evaluate(svm_predictions)
print(f"Area Under ROC: {lr_AUC:.4f}")
print(f'Area Under PR: {svm_auc_pr}')

The best threshold is -1.0, with a score of 1.7038786288105836
Accuracy: 0.608
Precision: 0.12149532710280374
Recall: 0.7647058823529411
F1 measure: 0.2096774193548387
Area Under ROC: 0.6662
Area Under PR: 0.09278374460088172


In [9]:
# -----------
# SVM WITH FIXED CLASS IMBALANCE
# -----------
# Step 1: Define SVM model with class weights
svm_class_imbalance = LinearSVC(
    featuresCol="features", 
    labelCol="financing_next_period", 
    weightCol="weight",  # Use the class weight column
    maxIter=100, 
    regParam=0.1
)

# Step 2: Create a pipeline to include preprocessing and SVM model
svm_class_imbalance_pipeline = Pipeline(stages=[
    industry_indexer,  # Index categorical column 'industry'
    maturity_indexer,  # Index categorical column 'maturity'
    ENC,               # One-hot encode index columns
    CV,                # Count vectorize 'locations' feature           
    *hints,
    sent_VA, 
    sent_scaler, 
    VA_final,
    svm_class_imbalance             # Train Linear SVM
])
# Step 4: Train the model
svm_class_imbalance = svm_class_imbalance_pipeline.fit(train_w)

# Step 5: Make predictions
svm_class_imbalance_predictions = svm_class_imbalance.transform(test_w)  # `test_s` contains standardized test data

# Step 6: Evaluate the model
eval_df = svm_class_imbalance_predictions.select('ticker', 'biweekly_period', 'financing_next_period', 'prediction', 'rawPrediction')

optimize_threshold(threshold_list, eval_df)
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="financing_next_period", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
svm_auc = evaluator_auc.evaluate(svm_class_imbalance_predictions)

evaluator_auc_pr = BinaryClassificationEvaluator(
    labelCol="financing_next_period", 
    metricName="areaUnderPR"
)
svm_auc_pr = evaluator_auc_pr.evaluate(svm_class_imbalance_predictions)

print(f'Area Under ROC: {svm_auc}')
print(f'Area Under PR: {svm_auc_pr}')

The best threshold is -2.0, with a score of 1.5564667740636327
Accuracy: 0.304
Precision: 0.08900523560209424
Recall: 1.0
F1 measure: 0.16346153846153844
Area Under ROC: 0.5925271396112094
Area Under PR: 0.0762603620752243


In [83]:
# -----------
# HYPERPARAMETER TUNING
# -----------
def walk_forward_eval(
    df_train,
    pipeline,
    time_col="biweekly_index",
    label_col="financing_next_period",
    init_train_end=20,     # last period in first training window
    val_len=3,             # number of future periods to validate on
    step=2,                # how much the training window moves forward
    embargo=0             # gap between train and validation (for lag features)
):
    """
    Performs expanding-window (walk-forward) evaluation.

    Returns:
        List of dicts with performance per window
    """
    evaluator_auc = BinaryClassificationEvaluator(
        labelCol="financing_next_period", 
        rawPredictionCol="rawPrediction", 
        metricName="areaUnderROC"
    )
    evaluator_auc_pr = BinaryClassificationEvaluator(
        labelCol="financing_next_period", 
        metricName="areaUnderPR"
    )

    results = []
    max_time = df_train.agg(F.max(time_col)).collect()[0][0]

    train_end = init_train_end

    while True:
        # Define validation window
        val_start = train_end + 1 + embargo
        val_end   = val_start + val_len - 1

        if val_end > max_time:
            break

        # Slice train / validation
        train_w = df_train.filter(F.col(time_col) <= train_end)
        val_w   = df_train.filter(
            (F.col(time_col) >= val_start) &
            (F.col(time_col) <= val_end)
        )

        # Fit model on past
        model = pipeline.fit(train_w)

        # Predict future
        pred = model.transform(val_w)

        # ---------- PR-AUC ----------
        auc = evaluator_auc.evaluate(pred)
        auc_pr = evaluator_auc_pr.evaluate(pred)

        # ---------- Recall / Precision at threshold ----------
        threshold = optimize_threshold(threshold_list, pred)

        evaluation_frame = custom_predictions(threshold, pred)
        accuracy, precision, recall, f1_measure = calculate_classification_metrics(evaluation_frame, 'financing_next_period', 'custom_prediction') 

        results.append({
            "train_end": int(train_end),
            "val_start": int(val_start),
            "val_end": int(val_end),
            "areaUnderPR": float(auc_pr),
            "areaUnderROC": float(auc),
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1_measure": float(f1_measure)
        })

        # Move window forward
        train_end += step

    return results


# Define the parameter grid for hyperparameter tuning
reg_grid = [0.001, 0.01, 0.05, 0.1, 0.2]
maxIter_grid = [50, 100, 200]

param_grid = []
for reg in reg_grid:
    for iter in maxIter_grid:
        param_grid.append({
            "regParam": reg,
            "maxIter": iter
        })

In [84]:
param_grid

[{'regParam': 0.001, 'maxIter': 50},
 {'regParam': 0.001, 'maxIter': 100},
 {'regParam': 0.001, 'maxIter': 200},
 {'regParam': 0.01, 'maxIter': 50},
 {'regParam': 0.01, 'maxIter': 100},
 {'regParam': 0.01, 'maxIter': 200},
 {'regParam': 0.05, 'maxIter': 50},
 {'regParam': 0.05, 'maxIter': 100},
 {'regParam': 0.05, 'maxIter': 200},
 {'regParam': 0.1, 'maxIter': 50},
 {'regParam': 0.1, 'maxIter': 100},
 {'regParam': 0.1, 'maxIter': 200},
 {'regParam': 0.2, 'maxIter': 50},
 {'regParam': 0.2, 'maxIter': 100},
 {'regParam': 0.2, 'maxIter': 200}]

In [ ]:
best = None
all_results = []

for i, params in enumerate(param_grid):
    print(f'Grid {i+1}')
    # Step 1: Define SVM model with class weights
    svm_class_imbalance = LinearSVC(
        featuresCol="features", 
        labelCol="financing_next_period", 
        weightCol="weight",  # Use the class weight column
        regParam=params['regParam'],
        maxIter=params['maxIter']
    )

    # Step 2: Create a pipeline to include preprocessing and SVM model
    svm_class_imbalance_pipeline = Pipeline(stages=[
        industry_indexer,  # Index categorical column 'industry'
        maturity_indexer,  # Index categorical column 'maturity'
        ENC,               # One-hot encode index columns
        CV,                # Count vectorize 'locations' feature           
        *hints,
        sent_VA, 
        sent_scaler, 
        VA_final,
        svm_class_imbalance               
    ])

    wf_rows = walk_forward_eval(   # your fixed-range version (0..47)
        df_train=train_w,
        pipeline=svm_class_imbalance_pipeline,
        time_col="biweekly_index",
        label_col="financing_next_period",
        init_train_end=20,   # train 0..20
        val_len=3,
        step=2,
        embargo=0
    )

    avg_pr_auc = sum(r["areaUnderPR"] for r in wf_rows) / len(wf_rows)
    avg_auc = sum(r["areaUnderROC"] for r in wf_rows) / len(wf_rows)
    avg_accuracy = sum(r["accuracy"] for r in wf_rows) / len(wf_rows)
    avg_precision = sum(r["precision"] for r in wf_rows) / len(wf_rows)
    avg_recall = sum(r["recall"] for r in wf_rows) / len(wf_rows)
    avg_f1 = sum(r["f1_measure"] for r in wf_rows) / len(wf_rows)

    score = avg_pr_auc + avg_auc + 0.6 * avg_accuracy + avg_precision + 3 * avg_recall + avg_f1

    all_results.append((params, score, avg_pr_auc, avg_auc, avg_accuracy, avg_precision, avg_recall, avg_f1))

    # OPTION A: prioritize PR-AUC, break ties with recall
    if (best is None) or (score > best[1]):
        best = (params, score, avg_pr_auc, avg_auc, avg_accuracy, avg_precision, avg_recall, avg_f1)

print("BEST LR PARAMS:", best[0])
print("Score:", best[1])
print("Avg walk-forward AreaUnderPR:", best[2])
print("Avg walk-forward AreaUnderROC", best[3])
print("Avg walk-forward Accuract", best[4])
print("Avg walk-forward Precision", best[5])
print("Avg walk-forward Recall", best[6])
print("Avg walk-forward F1 Score", best[7])

Grid 1


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-

KeyboardInterrupt: 